# Pixels to Predictions: DL Vision Challenge

**Model:** SmolVLM-500M-Instruct with LoRA fine-tuning
**Task:** Predict the correct answer index for science multiple-choice questions with image + text context
**Constraints:** ≤5M trainable params, free-tier compute, no external data

## 1. Imports and Configuration

All hyperparameters live in the `CFG` dataclass. `RunLogger` writes each run to an append-only JSONL log so experiments can be compared across runs.

In [ ]:
import os, json, math, random, gc
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    get_cosine_schedule_with_warmup,
    get_constant_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType

from run_logger import RunLogger


@dataclass
class CFG:
    # Paths
    data_dir: Path = Path("")
    output_dir: Path = Path("runs/lora_v2")
    run_log: Path = Path("runs/runs.jsonl")

    # Model
    model_id: str = "HuggingFaceTB/SmolVLM-500M-Instruct"
    img_size: int = 224

    # LoRA (must stay under 5M trainable params)
    lora_r: int = 24
    lora_alpha: int = 48
    lora_dropout: float = 0.05
    lora_target_modules: tuple = ("q_proj", "k_proj", "v_proj", "o_proj")

    # Training
    epochs: int = 9
    batch_size: int = 4
    grad_accum: int = 4
    lr: float = 2e-4
    weight_decay: float = 0.0
    warmup_ratio: float = 0.03
    text_token_budget: int = 600

    # Augmentation
    permute_choices: bool = True

    # Inference
    tta_k: int = 4

    # Eval
    val_subset: int = 400

    # Reproducibility
    seed: int = 42


RUN_NOTES = "dora, 60% val to train, e=9, cos schedule, total_steps=e11"

cfg = CFG()
cfg.output_dir.mkdir(parents=True, exist_ok=True)

random.seed(cfg.seed); np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed); torch.cuda.manual_seed_all(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"Device: {device}  dtype: {dtype}")

logger = RunLogger(cfg.run_log)
run_id = logger.start_run(cfg, notes=RUN_NOTES)

cfg.output_dir = cfg.output_dir / run_id
cfg.output_dir.mkdir(parents=True, exist_ok=True)

print(f"Logging to {cfg.run_log} as run_id={run_id}")


## 2. Load Data

Parse the `choices` column from JSON. Optionally fold a fraction of validation into training for the final run.

In [ ]:
train_df = pd.read_csv(cfg.data_dir / "train.csv")
val_df   = pd.read_csv(cfg.data_dir / "val.csv")
test_df  = pd.read_csv(cfg.data_dir / "test.csv")

for df in (train_df, val_df, test_df):
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")


In [ ]:
VAL_KEEP_FRAC = 0.40

new_val_df = (
    val_df
    .groupby("num_choices", group_keys=False)[val_df.columns.tolist()]
    .apply(lambda g: g.sample(frac=VAL_KEEP_FRAC, random_state=cfg.seed))
    .reset_index(drop=True)
)
val_to_train_df = val_df.drop(
    val_df.index[val_df["id"].isin(new_val_df["id"])]
).reset_index(drop=True)

train_df = pd.concat([train_df, val_to_train_df], ignore_index=True)
val_df = new_val_df

print(f"After mixing val into train:")
print(f"  Train: {len(train_df):,}  (added {len(val_to_train_df)} from val)")
print(f"  Val:   {len(val_df):,}    (kept {VAL_KEEP_FRAC:.0%} of original)")
print(f"  Test:  {len(test_df):,}")
print()
print("Val num_choices:", val_df["num_choices"].value_counts().sort_index().to_dict())
print("Train num_choices:", train_df["num_choices"].value_counts().sort_index().to_dict())


## 3. Prompt Format

The prompt ends in `Answer:` with a trailing space. We score the model's logits at the next-token position over the candidate letter tokens (` A`, ` B`, ...) — these are single tokens with the leading space.

In [ ]:
CHOICE_LETTERS = "ABCDEFGHIJ"


def build_prompt(question, choices, lecture=None, hint=None,
                 include_answer_letter=None):
    parts = []
    if isinstance(lecture, str) and lecture.strip():
        parts.append(lecture.strip())
    if isinstance(hint, str) and hint.strip():
        parts.append(hint.strip())
    ctx = "\n".join(parts)
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )
    s = "<image>\n"
    if ctx:
        s += f"Context:\n{ctx}\n\n"
    s += f"Question: {question}\nChoices:\n{choices_str}\nAnswer:"
    if include_answer_letter is not None:
        s += f" {include_answer_letter}"
    return s


## 4. Model and LoRA Setup

**LoRA configuration:**
- Targets language-model attention projections only (`q_proj, k_proj, v_proj, o_proj`)
- `r=24, alpha=48` (effective scaling 2.0×)
- DoRA decomposition for improved adaptation quality
- Vision tower frozen — would exceed the 5M-param cap

Gradient checkpointing is enabled to fit on a 16 GB GPU. The trainable-param assertion enforces the 5M cap.

In [ ]:
processor = AutoProcessor.from_pretrained(cfg.model_id)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

letter_token_ids = {}
for letter in CHOICE_LETTERS:
    ids = processor.tokenizer.encode(f" {letter}", add_special_tokens=False)
    letter_token_ids[letter] = ids[-1]

model = AutoModelForVision2Seq.from_pretrained(
    cfg.model_id,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)
if not torch.cuda.is_available():
    model.to(device)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# Target only the LM attention projections; vision tower stays frozen.
lm_module_names = [
    name for name, _ in model.named_modules()
    if "text_model.layers" in name
    and any(name.endswith(f".{m}") for m in cfg.lora_target_modules)
]

model = get_peft_model(model, LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    target_modules=lm_module_names,
    task_type=TaskType.CAUSAL_LM,
    use_dora=True,
))

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}  (cap: 5,000,000)")
assert trainable < 5_000_000, f"Over 5M cap (currently {trainable:,})"


## 5. Dataset

Choice-order permutation in `__getitem__` shuffles the answer position each epoch. This addresses position bias observed in zero-shot evaluation (5-choice questions concentrated on the A and E positions). Disabled for validation and test.

In [ ]:
class ScienceQADataset(Dataset):
    def __init__(self, df, data_dir, img_size, permute, is_train):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.permute = permute and is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        choices = list(row["choices"])
        answer = int(row["answer"]) if "answer" in row and not pd.isna(row["answer"]) else -1

        if self.permute:
            perm = list(range(len(choices)))
            random.shuffle(perm)
            choices = [choices[i] for i in perm]
            if answer >= 0:
                answer = perm.index(answer)

        img = Image.open(self.data_dir / row["image_path"]).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)

        return {
            "id": row["id"], "image": img,
            "question": row["question"], "choices": choices,
            "lecture": row.get("lecture"), "hint": row.get("hint"),
            "answer": answer, "num_choices": len(choices),
        }


train_ds = ScienceQADataset(train_df, cfg.data_dir, cfg.img_size, cfg.permute_choices, True)
val_ds   = ScienceQADataset(val_df,   cfg.data_dir, cfg.img_size, False, False)
test_ds  = ScienceQADataset(test_df,  cfg.data_dir, cfg.img_size, False, False)
print(f"train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}, permute={cfg.permute_choices}")


## 6. Training Collator

**Two key behaviors:**
- Pre-truncate the lecture text to fit a 600-token budget *before* the Idefics3 processor expands `<image>` into its ~1,088 image tokens. This avoids the post-expansion truncation slicing through the image-token block.
- Mask the loss to predict only the final answer-letter token. With ~5M trainable params on 3,109 examples, every gradient should target the actual classification objective.

In [ ]:
def _truncate_lecture(lecture, hint, question, choices, ans_letter, tok, budget):
    if not isinstance(lecture, str) or not lecture.strip():
        return lecture
    fixed = build_prompt(question, choices, lecture="", hint=hint,
                         include_answer_letter=ans_letter)
    fixed_tokens = len(tok.encode(fixed, add_special_tokens=False))
    available = budget - fixed_tokens
    if available <= 0:
        return ""
    lec_ids = tok.encode(lecture, add_special_tokens=False)
    if len(lec_ids) <= available:
        return lecture
    return tok.decode(lec_ids[:available], skip_special_tokens=True)


def train_collate(batch, processor, budget):
    images, prompts, ans_letters = [], [], []
    tok = processor.tokenizer
    for ex in batch:
        ans_letter = CHOICE_LETTERS[ex["answer"]]
        lecture = _truncate_lecture(ex["lecture"], ex["hint"], ex["question"],
                                     ex["choices"], ans_letter, tok, budget)
        full = build_prompt(ex["question"], ex["choices"], lecture, ex["hint"],
                            include_answer_letter=ans_letter)
        images.append(ex["image"]); prompts.append(full); ans_letters.append(ans_letter)

    enc = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    labels = torch.full_like(enc["input_ids"], -100)
    for i, letter in enumerate(ans_letters):
        tid = letter_token_ids[letter]
        seq = enc["input_ids"][i].tolist()
        for pos in range(len(seq) - 1, -1, -1):
            if seq[pos] == tid:
                labels[i, pos] = tid
                break
    enc["labels"] = labels
    return enc


_loader = DataLoader(train_ds, batch_size=2, shuffle=True,
                     collate_fn=lambda b: train_collate(b, processor, cfg.text_token_budget))
_b = next(iter(_loader))
assert (_b["labels"] != -100).sum().item() == 2, "Loss masking is wrong"
assert _b["input_ids"].shape[1] > 1000, "Image tokens not expanded"
print(f"Collator sanity passed. Batch shape: {_b['input_ids'].shape}")
del _loader, _b


## 7. Inference: Log-Likelihood Scoring + TTA

For each question we run a single forward pass and take the argmax over the logits at the answer-letter positions, restricted to the valid letters for that question's `num_choices`. This is faster and more robust than `generate()` parsing.

**Test-time augmentation:** K=4 random choice permutations per question. Per-choice probabilities are averaged after un-permuting, then argmaxed. Adds ~1–2pp on overall accuracy and ~10pp on the high-bias 5-choice category.

In [ ]:
@torch.inference_mode()
def score_one(ex):
    prompt = build_prompt(ex["question"], ex["choices"], ex["lecture"], ex["hint"],
                          include_answer_letter=None)
    inputs = processor(text=[prompt], images=[ex["image"]], return_tensors="pt")
    inputs = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in inputs.items()}
    out = model(**inputs)
    logits = out.logits[0, -1].float()
    valid = [letter_token_ids[CHOICE_LETTERS[i]] for i in range(ex["num_choices"])]
    scores = logits[valid].cpu().numpy()
    return int(np.argmax(scores)), scores


@torch.inference_mode()
def score_one_tta(ex, k, rng):
    n = ex["num_choices"]
    accum = np.zeros(n)
    for _ in range(k):
        perm = list(range(n)); rng.shuffle(perm)
        permuted = [ex["choices"][i] for i in perm]
        _, scores = score_one({**ex, "choices": permuted})
        probs = np.exp(scores - scores.max()); probs /= probs.sum()
        for j, orig in enumerate(perm):
            accum[orig] += probs[j]
    return int(np.argmax(accum)), accum / k


def evaluate(ds, max_examples=None, tta_k=1, desc="eval"):
    model.eval()
    n = len(ds) if not max_examples else min(len(ds), max_examples)
    indices = (list(range(n)) if not max_examples
               else random.Random(cfg.seed).sample(range(len(ds)), n))
    correct, total, by_nc = 0, 0, {}
    rng = random.Random(cfg.seed + 1)
    for i in tqdm(indices, desc=desc, leave=False):
        ex = ds[i]
        if ex["answer"] < 0:
            continue
        if tta_k > 1:
            pred, _ = score_one_tta(ex, tta_k, rng)
        else:
            pred, _ = score_one(ex)
        c = int(pred == ex["answer"])
        correct += c; total += 1
        by_nc.setdefault(ex["num_choices"], [0, 0])
        by_nc[ex["num_choices"]][0] += c
        by_nc[ex["num_choices"]][1] += 1
    return correct / max(total, 1), {k: v[0]/v[1] for k, v in sorted(by_nc.items())}, total


## 8. Optimizer and Schedule

AdamW with cosine schedule. The schedule is intentionally over-allocated (`total_steps = steps_per_epoch × 11`) while training only `cfg.epochs` (typically 9). This delays LR decay so the learning rate stays meaningful through the full training run.

In [ ]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=cfg.lr, weight_decay=cfg.weight_decay)

steps_per_epoch = math.ceil(len(train_ds) / (cfg.batch_size * cfg.grad_accum))

# Schedule for 11 epochs but train for cfg.epochs (9) to keep LR meaningful at peak.
total_steps = steps_per_epoch * 11
warmup_steps = max(1, int(total_steps * cfg.warmup_ratio))

scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"Steps/epoch: {steps_per_epoch}  Total scheduled: {total_steps}  Warmup: {warmup_steps}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU free: {free/1e9:.2f} / {total/1e9:.2f} GB")


## 9. Training Loop

Per-epoch validation uses K=1 (no TTA) for speed. The full TTA-K=4 evaluation runs once after training. The best checkpoint is saved by validation accuracy, not by loss — they diverge on classification tasks.

In [ ]:
train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size, shuffle=True,
    collate_fn=lambda b: train_collate(b, processor, cfg.text_token_budget),
    num_workers=2, pin_memory=True,
)

best_acc = 0.0
for epoch in range(cfg.epochs):
    model.train()
    running_loss, n_steps = 0.0, 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{cfg.epochs}")
    for step, batch in enumerate(pbar):
        batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
        out = model(**batch)
        (out.loss / cfg.grad_accum).backward()
        if (step + 1) % cfg.grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
        running_loss += out.loss.item(); n_steps += 1
        if n_steps % 20 == 0:
            pbar.set_postfix(loss=running_loss/n_steps, lr=scheduler.get_last_lr()[0])

    avg_loss = running_loss / max(n_steps, 1)
    acc, by_nc, n_eval = evaluate(
        val_ds, max_examples=cfg.val_subset or None, tta_k=1,
        desc=f"eval e{epoch+1}",
    )
    print(f"\n[epoch {epoch+1}] loss={avg_loss:.4f}  val_acc={acc:.4f} (n={n_eval})")
    for nc, a in by_nc.items():
        print(f"   {nc}-choice: {a:.4f}")

    logger.log_epoch(run_id, epoch=epoch+1, loss=avg_loss, val_acc=acc,
                     by_num_choices=by_nc, lr=scheduler.get_last_lr()[0])

    if acc > best_acc:
        best_acc = acc
        save_path = cfg.output_dir / "best"
        model.save_pretrained(save_path)
        print(f"   ✓ saved best to {save_path}  (acc={best_acc:.4f})")

print(f"\nBest val acc (no TTA): {best_acc:.4f}")


## 10. Final Evaluation and Submission

Runs TTA-K=4 on the full validation set, generates test predictions, validates the submission format, and writes `submission.csv`.

In [ ]:
tta_acc, tta_by_nc, tta_n = evaluate(
    val_ds, max_examples=None, tta_k=cfg.tta_k,
    desc=f"final TTA k={cfg.tta_k}",
)
print(f"TTA k={cfg.tta_k} val acc (n={tta_n}): {tta_acc:.4f}")
for nc, a in tta_by_nc.items():
    print(f"   {nc}-choice: {a:.4f}")

rng = random.Random(cfg.seed + 2)
rows = []
model.eval()
for i in tqdm(range(len(test_ds)), desc="predict test"):
    ex = test_ds[i]
    pred, _ = score_one_tta(ex, cfg.tta_k, rng) if cfg.tta_k > 1 else score_one(ex)
    rows.append({"id": ex["id"], "answer": int(pred)})
submission = pd.DataFrame(rows)

assert list(submission.columns) == ["id", "answer"]
assert len(submission) == len(test_df) and submission["id"].is_unique
assert set(submission["id"]) == set(test_df["id"])
assert submission["answer"].dtype.kind == "i"
for _, r in submission.merge(test_df[["id", "num_choices"]], on="id").iterrows():
    assert 0 <= r["answer"] < r["num_choices"]

sub_path = cfg.output_dir / "submission.csv"
submission.to_csv(sub_path, index=False)
print(f"Wrote {sub_path}  ({len(submission)} rows)")
print("\nPrediction distribution:")
print(submission["answer"].value_counts().sort_index())

logger.finalize(run_id, best_val_acc=best_acc,
                tta_val_acc=tta_acc, tta_by_num_choices=tta_by_nc)
print(f"\nFinalized run_id={run_id}")


## 11. Record Leaderboard Score

After Kaggle submission, attach the public leaderboard score to this run's record in `runs.jsonl`.

In [ ]:
# Run after Kaggle submission with the public LB score from the leaderboard.
# logger.add_lb_score(run_id, public_lb=0.86519)

logger.show_table()
